# Linear Mixed Models (RQ3)

In [ ]:
import warnings
import itertools

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

##
warnings.filterwarnings("ignore")

## Load and merge data

In [ ]:
demo = pd.read_csv("../DATA/demo.csv")
blocks = pd.read_pickle("../DATA/blocks.pkl")

demo["date"] = pd.to_datetime(demo["date"])
demo["age"] = demo["date"].dt.year - demo["birth_year"]


freq_map = {"never": 0, "rarely": 1, "Several times a week": 2, "Daily": 3}
demo["driving_frequency_num"] = demo["driving_frequency"].map(freq_map)


df = blocks.merge(demo, on="participant_id", how="left", validate="many_to_one")

REFERENCE_CONDITION = "Nelo"  # baseline condition
df["condition"] = pd.Categorical(df["condition"], categories=[REFERENCE_CONDITION] + [c for c in df["condition"].unique() if c != REFERENCE_CONDITION])

print(f"Participants: {df['participant_id'].nunique()}, rows (participant x block): {df.shape[0]}")
df[["participant_id", "condition", "trust", "understanding", "nasa"]].head()


## candidate moderators

Personality (BFI dimensions), need for cognition, technology readiness, age, driving frequency, and automation experience.

In [ ]:
continuous_predictors = [
    "bfi_extraversion",
    "bfi_agreeableness",
    "bfi_conscientiousness",
    "bfi_neuroticism",
    "bfi_openness",
    "nfc_score",
    "tri_score",
    "age",
    "driving_frequency_num",
]

categorical_predictors = {
    "automation_experience": "No",  # reference level
}

all_candidate_predictors = continuous_predictors + list(categorical_predictors)
print(f"{len(all_candidate_predictors)} candidate moderators:", all_candidate_predictors)

## standardise continuous predictors

Continuous predictors (including the retained BFI dimensions, need for cognition, technology readiness, age, and driving frequency) are $z$-standardised across participants. Categorical moderators (`automation_experience`) and `condition` are left untransformed.

In [ ]:
z_map = {}
for var in continuous_predictors:
    z_name = f"z_{var}"
    mu, sd = demo[var].mean(), demo[var].std(ddof=1)
    demo[z_name] = (demo[var] - mu) / sd
    z_map[var] = z_name

# re-merge so df has the z-scored columns
df = blocks.merge(demo, on="participant_id", how="left", validate="many_to_one")
df["condition"] = pd.Categorical(df["condition"], categories=[REFERENCE_CONDITION] + [c for c in df["condition"].unique() if c != REFERENCE_CONDITION])

predictor_terms = {}
for var in continuous_predictors:
    predictor_terms[var] = z_map[var]
for var, ref in categorical_predictors.items():
    predictor_terms[var] = f"C({var}, Treatment(reference='{ref}'))"

predictor_terms


## Model fitting helpers

For each outcome and each candidate predictor:

1. **Univariate additive model**: `DV ~ Condition + Predictor + (1 | Participant)`
2. **Interaction model**: `DV ~ Condition * Predictor + (1 | Participant)`
3. A **likelihood ratio test** (both models fit by maximum likelihood, `reml=False`) compares the interaction model against the additive model.
4. **Wald z-statistics** for every fixed-effect coefficient are extracted from both models.

In [ ]:
CONDITION_TERM = f"C(condition, Treatment(reference='{REFERENCE_CONDITION}'))"

def fit_mixedlm(formula, data, group_col="participant_id"):

    last_err = None
    for method in ("lbfgs", "bfgs", "cg", "powell"):
        try:
            model = smf.mixedlm(formula, data=data, groups=data[group_col])
            result = model.fit(reml=False, method=method, maxiter=200)
            if result.converged:
                return result, method
            last_err = "did not converge"
        except Exception as e:
            last_err = str(e)
    raise RuntimeError(f"Model failed to fit ({formula}): {last_err}")


def fe_table(result, dv, predictor, model_type):

    fe_names = list(result.fe_params.index)
    out = pd.DataFrame({
        "outcome": dv,
        "predictor": predictor,
        "model": model_type,
        "term": fe_names,
        "estimate": result.fe_params.values,
        "se": result.bse_fe.values,
        "z": result.tvalues[fe_names].values,
        "p": result.pvalues[fe_names].values,
    })
    return out


def likelihood_ratio_test(reduced_result, full_result, reduced_model, full_model):
    lr_stat = 2 * (full_result.llf - reduced_result.llf)
    df_diff = full_model.k_fe - reduced_model.k_fe
    lr_stat = max(lr_stat, 0.0)
    p_value = stats.chi2.sf(lr_stat, df_diff) if df_diff > 0 else np.nan
    return lr_stat, df_diff, p_value


## Run the stepwise procedure for all outcomes x predictors

In [ ]:
outcomes = ["trust", "understanding", "nasa"]

univariate_rows = []
interaction_rows = []
lrt_rows = []
fit_failures = []

for dv, predictor in itertools.product(outcomes, all_candidate_predictors):
    term = predictor_terms[predictor]
    add_formula = f"{dv} ~ {CONDITION_TERM} + {term}"
    int_formula = f"{dv} ~ {CONDITION_TERM} * {term}"

    try:
        add_model = smf.mixedlm(add_formula, data=df, groups=df["participant_id"])
        add_result, add_method = fit_mixedlm(add_formula, df)

        int_model = smf.mixedlm(int_formula, data=df, groups=df["participant_id"])
        int_result, int_method = fit_mixedlm(int_formula, df)
    except RuntimeError as e:
        fit_failures.append({"outcome": dv, "predictor": predictor, "error": str(e)})
        continue

    univariate_rows.append(fe_table(add_result, dv, predictor, "univariate additive"))
    interaction_rows.append(fe_table(int_result, dv, predictor, "interaction"))

    lr_stat, df_diff, p_value = likelihood_ratio_test(add_result, int_result, add_model, int_model)
    lrt_rows.append({
        "outcome": dv, "predictor": predictor,
        "chi2": lr_stat, "df": df_diff, "p": p_value,
        "converged_additive": add_result.converged, "converged_interaction": int_result.converged,
    })

univariate_results = pd.concat(univariate_rows, ignore_index=True) if univariate_rows else pd.DataFrame()
interaction_results = pd.concat(interaction_rows, ignore_index=True) if interaction_rows else pd.DataFrame()
lrt_results = pd.DataFrame(lrt_rows)

if fit_failures:
    print(f"{len(fit_failures)} model(s) failed to converge/fit - see `fit_failures`:")
    display(pd.DataFrame(fit_failures))
else:
    print("All models fit successfully.")


## Results

### Univariate additive models (`DV ~ Condition + Predictor`)
One row per fixed-effect term, per outcome x predictor combination.

In [ ]:
univariate_results_sorted = univariate_results.sort_values(["outcome", "predictor", "term"]).reset_index(drop=True)
univariate_results_sorted


### Interaction models (`DV ~ Condition * Predictor`)

In [ ]:
interaction_results_sorted = interaction_results.sort_values(["outcome", "predictor", "term"]).reset_index(drop=True)
interaction_results_sorted


### Likelihood ratio tests (interaction vs. additive model)

In [ ]:
lrt_results_sorted = lrt_results.sort_values(["outcome", "p"]).reset_index(drop=True)
lrt_results_sorted


### Quick-scan summary

For each outcome, the predictor's own Wald z/p (from the additive model) and the condition x predictor interaction LRT p-value, side by side.

In [ ]:
def predictor_main_effect_row(res_df, predictor):

    mask = res_df["term"].str.contains(predictor, regex=False) & ~res_df["term"].str.contains(":", regex=False) & ~res_df["term"].str.contains("condition", regex=False)
    return res_df[mask]

summary_rows = []
for dv in outcomes:
    for predictor in all_candidate_predictors:
        uni = univariate_results[(univariate_results.outcome == dv) & (univariate_results.predictor == predictor)]
        main_rows = predictor_main_effect_row(uni, predictor)
        lrt_row = lrt_results[(lrt_results.outcome == dv) & (lrt_results.predictor == predictor)]
        for _, r in main_rows.iterrows():
            summary_rows.append({
                "outcome": dv,
                "predictor": predictor,
                "term": r["term"],
                "z_main_effect": r["z"],
                "p_main_effect": r["p"],
                "LRT_chi2_interaction_vs_additive": lrt_row["chi2"].values[0] if len(lrt_row) else np.nan,
                "LRT_df": lrt_row["df"].values[0] if len(lrt_row) else np.nan,
                "LRT_p_interaction_vs_additive": lrt_row["p"].values[0] if len(lrt_row) else np.nan,
            })

summary_table = pd.DataFrame(summary_rows).sort_values(["outcome", "LRT_p_interaction_vs_additive"]).reset_index(drop=True)
summary_table


## save results

In [ ]:
univariate_results_sorted.to_csv("univariate_model_results.csv", index=False)
interaction_results_sorted.to_csv("interaction_model_results.csv", index=False)
lrt_results_sorted.to_csv("lrt_results.csv", index=False)
summary_table.to_csv("moderator_summary.csv", index=False)
print("Saved: univariate_model_results.csv, interaction_model_results.csv, lrt_results.csv, moderator_summary.csv")
